# DLPFC — all 18 pairs
Latest local main function; coordinates converted to μm with 100 μm = 138 input pixels. Existing feature costs are reused. Evaluation retains historical GPR radius 300 in original input coordinate units. Run in a fresh kernel. Previous results are preserved under previous_results/.

In [ ]:
from pathlib import Path
import json
import os
import sys

os.environ.setdefault("NUMBA_CACHE_DIR", "/private/tmp/raftup_cutoff_numba")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/raftup_cutoff_mpl")

import numpy as np
import torch

RAFTUP_SRC = Path("/Users/salovjade/Library/CloudStorage/Dropbox/boatwhere/raftup_repo/src")
REPRODUCE_ROOT = Path("/Users/salovjade/Library/CloudStorage/Dropbox/RAFTUP_after0320/raftup_reproduce")
sys.path.insert(0, str(RAFTUP_SRC))

from raftup import _load_data, _metrics_two_gpr
from raftup._cutoff_selection import run_automated_alignment

import hashlib, inspect
from raftup import _cutoff_selection as current_cs
source_root = Path("/Users/salovjade/Library/CloudStorage/Dropbox/boatwhere/raftup_repo/src/raftup")
assert Path(current_cs.__file__).resolve() == (source_root / "_cutoff_selection.py").resolve(), "Restart kernel and import local repo"
assert inspect.signature(current_cs.run_automated_alignment).parameters["technology"].default == "dlpfc"
assert "pixels_per_100um" in inspect.signature(current_cs.run_automated_alignment).parameters, "Restart kernel: outdated main function"
SOURCE_FINGERPRINTS = {str(p.relative_to(source_root)): hashlib.sha256(p.read_bytes()).hexdigest()
                       for p in sorted(source_root.rglob("*")) if p.is_file() and p.suffix in {".py", ".pkl"}}

OUT = Path('/Users/salovjade/Library/CloudStorage/Dropbox/raftup_revision_828/all_exp/dlpfc')
PAIRS = {'0708': ['151507', '151508'], '0709': ['151507', '151509'], '0710': ['151507', '151510'], '0809': ['151508', '151509'], '0810': ['151508', '151510'], '0910': ['151509', '151510'], '6970': ['151669', '151670'], '6971': ['151669', '151671'], '6972': ['151669', '151672'], '7071': ['151670', '151671'], '7072': ['151670', '151672'], '7172': ['151671', '151672'], '7374': ['151673', '151674'], '7375': ['151673', '151675'], '7376': ['151673', '151676'], '7475': ['151674', '151675'], '7476': ['151674', '151676'], '7576': ['151675', '151676']}
SELECTED_PAIRS = list(PAIRS)


In [ ]:
def run_pair(PAIR_LABEL):
    SECTION_A, SECTION_B = PAIRS[PAIR_LABEL]
    sliceA = _load_data.load_DLPFC(section_id=SECTION_A)
    sliceB = _load_data.load_DLPFC(section_id=SECTION_B)

    FEATURE_COST_PATH = (
        REPRODUCE_ROOT
        / "0_preprocessing/feature_cost_in_DLPFC/cost_mac_reproducible_robust_somde"
        / f"{SECTION_A}_{SECTION_B}_somde_cost_matrix.pt"
    )
    feature_cost = torch.load(FEATURE_COST_PATH, map_location="cpu", weights_only=False)
    print("Slice sizes:", len(sliceA), len(sliceB))
    print("Feature cost:", tuple(feature_cost.shape))

    alignment_matrix, decision = run_automated_alignment(
        sliceA,
        sliceB,
        feature_cost,
        technology="dlpfc",
        pixels_per_100um=138,
    )

    print("Classifier:", decision["classifier_prediction"])
    print("P(similar):", round(decision["p_similar"], 4))
    print("Selected attempt:", decision["selected_attempt"])
    print("Alignment shape:", alignment_matrix.shape)

    labels = np.concatenate([
        np.asarray(sliceA.obs["original_clusters"]),
        np.asarray(sliceB.obs["original_clusters"]),
    ])

    scores = {
        "LAA": float(
            _metrics_two_gpr.cal_layer_based_alignment_result_full_skip_all_zero(
                alignment_matrix, labels
            )[0]
        ),
        "GPR_300": float(
            _metrics_two_gpr.GPR_original(
                alignment_matrix.copy(),
                sliceA.obsm["spatial"],
                sliceB.obsm["spatial"],
                dis_cut=300,
                P_cut=1e-19,
            )
        ),
        "coverage_A": float(np.any(alignment_matrix != 0, axis=1).mean()),
        "coverage_B": float(np.any(alignment_matrix != 0, axis=0).mean()),
    }

    OUTPUT_DIR = OUT / PAIR_LABEL
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    np.save(OUTPUT_DIR / f"alignment_matrix_{PAIR_LABEL}.npy", alignment_matrix)
    (OUTPUT_DIR / "decision.json").write_text(json.dumps(decision, indent=2))
    (OUTPUT_DIR / "scores.json").write_text(json.dumps(scores, indent=2))

    print(json.dumps(scores, indent=2))
    print("Saved to:", OUTPUT_DIR.resolve())

    (OUTPUT_DIR / "source_fingerprints.json").write_text(json.dumps(SOURCE_FINGERPRINTS, indent=2))

    return dict(pair=PAIR_LABEL, **scores)

In [ ]:
import pandas as pd
results = []
for pair in SELECTED_PAIRS:
    results.append(run_pair(pair))
    pd.DataFrame(results).to_csv(OUT / "all_pair_metrics.csv", index=False)
display(pd.DataFrame(results))